In [32]:
import os
import re
import time
import shutil
import hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from html.parser import HTMLParser
from typing import List, Tuple, Dict, Any
from pathlib import Path


In [33]:
def ensure_dir(path):
    if not os.path.exists(path):
        print(f"{path} does not exist. Creating it...")
        os.makedirs(path)

In [34]:
# Base folder that contains c1_juris, c2_juris, t2_juris subfolders
BASE_DIR = os.getcwd()
BASE_DIR = Path(BASE_DIR)  # ← change me

# Source folders to scan under BASE_DIR
SOURCES = ["c1_juris", "c2_juris", "t2_juris"]

# Output directory for the Excel files (defaults to BASE_DIR)
OUT_DIR = BASE_DIR

In [35]:
# Cell 2 — Helpers: extract visible text and article mentions (numbers + labels)
try:
    from bs4 import BeautifulSoup
    _HAS_BS4 = True
except Exception:
    _HAS_BS4 = False

# Match common variants: "ARTICLE 101", "Article 82", "Art. 81", "Art 101 TFEU", etc.
# Group 1 captures the number.
ARTICLE_MENTION_RE = re.compile(
    r"\b(?:ART(?:\.|ICLE)?)\s*[:\.]?\s*(\d{1,3})\b",
    re.IGNORECASE
)

def read_text(path: Path) -> str:
    """Read file as text; if HTML, extract visible text; else fallback."""
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            raw = path.read_text(encoding=enc, errors="ignore")
            break
        except Exception:
            raw = ""
    if not raw:
        return ""
    if _HAS_BS4 and (path.suffix.lower() in {".html", ".htm"} or "<html" in raw.lower()):
        try:
            soup = BeautifulSoup(raw, "html.parser")
            body = soup.find(id="TexteOnly") or soup.body or soup
            text = body.get_text(" ", strip=True)
            return re.sub(r"\s+", " ", text)
        except Exception:
            pass
    # fallback: strip tags crudely
    text = re.sub(r"<[^>]+>", " ", raw)
    return re.sub(r"\s+", " ", text)

def find_article_mentions(text: str):
    """
    Return:
      - found (bool),
      - numbers (sorted unique list of ints),
      - labels (string like 'Article 81; Article 102')
    """
    nums = []
    for m in ARTICLE_MENTION_RE.finditer(text):
        try:
            n = int(m.group(1))
            if 1 <= n <= 200:  # guard against garbage
                nums.append(n)
        except Exception:
            continue
    uniq = sorted(set(nums))
    labels = "; ".join(f"Article {n}" for n in uniq)
    return (len(uniq) > 0), uniq, labels

In [36]:
# Cell 3 — Scan one source and return DataFrame with filename + found + details
def scan_source(source: str) -> pd.DataFrame:
    src_path = BASE_DIR / source
    rows = []
    if not src_path.exists():
        return pd.DataFrame(columns=["filename", "found_article", "article_numbers", "article_labels"])
    for root, _, files in os.walk(src_path):
        for fname in files:
            fpath = Path(root) / fname
            # Skip obvious binaries
            if fpath.suffix.lower() in {".png", ".jpg", ".jpeg", ".gif"}:
                rows.append({
                    "filename": fname,
                    "found_article": False,
                    "article_numbers": [],
                    "article_labels": ""
                })
                continue
            try:
                text = read_text(fpath)
                found, nums, labels = find_article_mentions(text)
            except Exception:
                found, nums, labels = False, [], ""
            rows.append({
                "filename": fname,
                "found_article": bool(found),
                "article_numbers": nums,   # list in memory; will be serialized in CSV as string
                "article_labels": labels,  # human-readable summary
            })
    return (
        pd.DataFrame(rows)
        .sort_values("filename")
        .reset_index(drop=True)
    )

In [37]:
# Cell 4 — Run all sources and write CSVs (files_{source}.csv)
for src in SOURCES:
    df = scan_source(src)
    out_csv = OUT_DIR / f"files_{src}.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"Wrote {len(df):>5} rows → {out_csv}")

Wrote  3396 rows → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_c1_juris.csv
Wrote 11861 rows → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_c2_juris.csv
Wrote 18399 rows → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_t2_juris.csv


In [64]:
# Optional Cell 5 — add competition flag and re-write CSVs
COMP_ARTICLES = {81, 82, 85, 86, 90, 101, 102, 106}

for src in SOURCES:
    df = pd.read_csv(OUT_DIR / f"files_{src}.csv")
    # Parse article_numbers back if needed (they'll be strings after CSV write)
    def to_list(x):
        if pd.isna(x) or not str(x).strip():
            return []
        # handle like "[81, 101]" or "81; 101" etc.
        s = str(x)
        # try eval-like parsing safely
        nums = re.findall(r"\d{1,3}", s)
        return sorted(set(int(n) for n in nums))
    nums_col = df["article_numbers"].apply(to_list)
    df["contains_competition_article"] = nums_col.apply(lambda lst: any(n in COMP_ARTICLES for n in lst))
    df.to_csv(OUT_DIR / f"files_{src}.csv", index=False, encoding="utf-8")
    print(f"Updated competition flag → {OUT_DIR / f'files_{src}.csv'}")

Updated competition flag → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_c1_juris.csv
Updated competition flag → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_c2_juris.csv
Updated competition flag → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\files_t2_juris.csv


In [66]:
# --- Cell 1 replacement: handle multiple sources instead of one folder ---

from pathlib import Path
import re, json, html, sys, pandas as pd

try:
    from bs4 import BeautifulSoup
    _HAS_BS4 = True
except Exception:
    _HAS_BS4 = False

# ======== CONFIG =========
SOURCES = ["c1_juris", "c2_juris", "t2_juris"]
BASE_DIR = Path.cwd()                     # parent directory containing those folders
OUT_FILE  = BASE_DIR / "scan_mentions.csv"

# ======== REGEX PATTERNS (same as before) =========
ARTICLE_REGEX = re.compile(
    r"""
    (?:^|(?<=\W))                                      # start or non-word before
    (?:ex\s+)?                                         # optional 'ex ' (ex Article 81 EC)
    (?:Article|Articles|Arts?\.?|Artikel|Art\.?|Art)   # prefixes incl. 'Arts.'
    \s*                                                # allow 'Art.101' or 'Art 101'
    (?P<num_block>
        (?:                                             # first number (maybe bracketed)
            \[\s*\d{1,3}(?:\s*\([^)]+\))*\s*
                (?:TFEU|AEUV|EC|EG|EEC|EWG|EU|EUV|EGV)?\s*\]   # [101(1) TFEU]
          | \d{1,3}(?:\s*\([0-9a-zA-Z]+\))*                     # 101(1)(a)
                (?:\s*(?:TFEU|AEUV|EC|EG|EEC|EWG|EU|EUV|EGV))?  # optional treaty tag
        )
        (?:\s*                                                  # additional joined numbers
            (?:,|;|and|or|und|oder|&|[-\u2013])\s*              # incl. hyphen/en dash
            (?:\[\s*\d{1,3}(?:\s*\([^)]+\))*\s*
                    (?:TFEU|AEUV|EC|EG|EEC|EWG|EU|EUV|EGV)?\s*\]
              | \d{1,3}(?:\s*\([0-9a-zA-Z]+\))*
                    (?:\s*(?:TFEU|AEUV|EC|EG|EEC|EWG|EU|EUV|EGV))?
            )
        )*
    )
    (?:\s*of\s+the\s+(?:                                # optional long suffix
        (?:EEC|EC|EU)\s+Treaty
        |Treaty|TFEU|EC|EU|EUV|AEUV
    ))?
    (?=\W|$)                                            # safe end
    """,
    re.IGNORECASE | re.VERBOSE,
)


# ======== KEYWORDS (extended German terms) =========
KEYWORDS = {
    "cartel": re.compile(r"\b(cartel|kartell)\b", re.IGNORECASE),
    "antitrust": re.compile(r"\b(antitrust|kartellrecht)\b", re.IGNORECASE),
    "competition_law": re.compile(r"\b(wettbewerbsrecht)\b", re.IGNORECASE),
    "competition": re.compile(r"\b(wettbewerb)\b", re.IGNORECASE),
    "dominance": re.compile(r"\b(marktbeherrschend(?:e[nrms]?|er|en)?)\b", re.IGNORECASE),
}

COMP_ARTICLES = {81, 82, 85, 86, 101, 102, 106, 65, 66}

def _normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()



In [67]:
# --- Cell 2: text extraction & context helpers ---

def read_text_from_file(p: Path, max_bytes: int = 8_000_000) -> str:
    """
    Reads HTML/TXT into plain text. For HTML, strips tags (BeautifulSoup if available).
    Returns "" for unsupported types; wire in a PDF extractor later if needed.
    """
    if not p or not p.exists() or not p.is_file():
        return ""

    try:
        suf = p.suffix.lower()
        if suf in {".html", ".htm"}:
            raw = p.read_bytes()[:max_bytes]
            for enc in ("utf-8", "cp1252", "latin-1"):
                try:
                    text = raw.decode(enc, errors="ignore")
                    break
                except Exception:
                    continue
            else:
                text = raw.decode("utf-8", errors="ignore")

            if _HAS_BS4:
                soup = BeautifulSoup(text, "lxml") if "lxml" in sys.modules else BeautifulSoup(text, "html.parser")
                text = soup.get_text(" ")
            else:
                text = re.sub(r"<[^>]+>", " ", text)

            return _normalize_ws(html.unescape(text))

        if suf in {".txt", ".md"}:
            return _normalize_ws(p.read_text(encoding="utf-8", errors="ignore")[:max_bytes])

        # Optional: add PDF parsing with pdfminer.six or pypdf
        return ""
    except Exception:
        return ""

def infer_lang_from_path(p: Path) -> str | None:
    """
    Tries to infer language from typical EUR-Lex path hints:
      - '/EN/TXT' or '/DE/TXT'
      - filename suffix like *_EN.html or *_DE.html
    """
    s = str(p).upper()
    # URL-style segments
    m = re.search(r"/([A-Z]{2})/TXT", s)
    if m:
        return m.group(1)
    # filename suffix
    m = re.search(r"_([A-Z]{2})\.(HTML?|TXT)$", s)
    if m:
        return m.group(1)
    return None

def extract_context_snippets(text: str, regex: re.Pattern, window: int = 80, max_snips: int = 5) -> list[str]:
    """
    Returns short context snippets around matches, for quick auditing in the CSV.
    """
    if not text:
        return []
    snips = []
    for i, m in enumerate(regex.finditer(text)):
        if i >= max_snips:
            break
        a, b = m.span()
        start = max(0, a - window)
        end   = min(len(text), b + window)
        snip = text[start:end].replace("\n", " ")
        snips.append(_normalize_ws(snip))
    return snips


In [68]:
CASE_NO_REGEX = re.compile(
    r"""
    \b
    (?P<prefix>COMP|IV|AT)        # exact uppercase prefixes
    /                             # literal slash
    (?P<num>\d+(?:\s*\d+)*(?:\.\d+)?)  # digits, possibly with internal spaces, optional .digits
    \b
    """,
    re.VERBOSE,  # case-sensitive, allows whitespace and comments
)


In [69]:
def extract_mentions(text: str) -> dict:
    """
    Extract:
      - treaty article mentions + snippets
      - competition-law keywords + snippets
      - IV/AT case numbers + snippets
    """
    if not text:
        return {
            "found_article": False,
            "article_matches": [],
            "article_snippets": [],
            "found_keywords": False,
            "keywords_present": [],
            "keyword_matches": [],
            "keyword_snippets": [],
            "found_case_no": False,
            "case_matches": [],
            "case_snippets": [],
        }

    # --- Articles ---
    article_matches = []
    seen_article = set()
    for m in ARTICLE_REGEX.finditer(text):
        s = m.group(0).strip()
        key = s.lower()
        if key not in seen_article:
            seen_article.add(key)
            article_matches.append(s)

    article_snippets = extract_context_snippets(text, ARTICLE_REGEX, window=80, max_snips=5) \
        if article_matches else []

    # --- Keywords ---
    keywords_present = []
    kw_hits_all = []
    first_kw_regex = None

    for key, rx in KEYWORDS.items():
        hits = [m.group(0).strip() for m in rx.finditer(text)]
        if hits:
            keywords_present.append(key)
            kw_hits_all.extend(hits)
            if first_kw_regex is None:
                first_kw_regex = rx

    # dedupe keyword tokens but keep original spelling
    keyword_matches = []
    seen_kw = set()
    for h in kw_hits_all:
        k = h.lower()
        if k not in seen_kw:
            seen_kw.add(k)
            keyword_matches.append(h)

    keyword_snippets = (
        extract_context_snippets(text, first_kw_regex, window=80, max_snips=5)
        if first_kw_regex is not None
        else []
    )

    # --- Case numbers (IV / AT) ---
    case_matches = []
    seen_case = set()
    for m in CASE_NO_REGEX.finditer(text):
        # normalize to standard form: e.g. "IV 30.004"
        prefix = m.group("prefix").upper()
        num = m.group("num")
        normalized = f"{prefix} {num}"
        key = normalized.lower()
        if key not in seen_case:
            seen_case.add(key)
            case_matches.append(normalized)

    case_snippets = (
        extract_context_snippets(text, CASE_NO_REGEX, window=80, max_snips=5)
        if case_matches
        else []
    )

    return {
        "found_article": bool(article_matches),
        "article_matches": article_matches,
        "article_snippets": article_snippets,

        "found_keywords": bool(keyword_matches),
        "keywords_present": keywords_present,
        "keyword_matches": keyword_matches,
        "keyword_snippets": keyword_snippets,

        "found_case_no": bool(case_matches),
        "case_matches": case_matches,
        "case_snippets": case_snippets,
    }


In [70]:
# --- Cell 4 replacement: scan multiple source folders and flag competition articles ---

def extract_article_numbers(article_matches: list[str]) -> set[int]:
    """Extract numeric parts (e.g., 101, 102) from article strings."""
    nums = set()
    for a in article_matches or []:
        m = re.search(r"\b(\d{1,3})\b", a)
        if m:
            try:
                nums.add(int(m.group(1)))
            except ValueError:
                pass
    return nums


records, scanned = [], 0

for src in SOURCES:
    src_dir = BASE_DIR / src
    if not src_dir.exists():
        print(f"⚠️  Skipping missing folder: {src_dir}")
        continue

    for p in src_dir.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".html", ".htm", ".txt"}:
            continue

        scanned += 1
        text = read_text_from_file(p)
        res = extract_mentions(text)

        # extract article numbers & competition flag
        article_nums = extract_article_numbers(res.get("article_matches"))
        mentions_comp_article = any(num in COMP_ARTICLES for num in article_nums)

        rec = {
            "source": src,
            "path": str(p.resolve()),
            "filename": p.name,
            "ext": p.suffix.lower(),
            "size_bytes": p.stat().st_size if p.exists() else None,
            "lang_inferred": infer_lang_from_path(p),

            "found_article": res["found_article"],
            "found_keywords": res["found_keywords"],
            "found_case_no": res["found_case_no"],
            "mentions_comp_article": mentions_comp_article,

            "first_article_mention": res["article_matches"][0] if res["article_matches"] else None,
            "first_keyword_mention": res["keyword_matches"][0] if res["keyword_matches"] else None,
            "first_case_mention": res["case_matches"][0] if res["case_matches"] else None,

            "article_matches_json": json.dumps(res["article_matches"], ensure_ascii=False),
            "article_snippets_json": json.dumps(res["article_snippets"], ensure_ascii=False),
            "keywords_present_json": json.dumps(res["keywords_present"], ensure_ascii=False),
            "keyword_matches_json": json.dumps(res["keyword_matches"], ensure_ascii=False),
            "keyword_snippets_json": json.dumps(res["keyword_snippets"], ensure_ascii=False),

            "case_matches_json": json.dumps(res["case_matches"], ensure_ascii=False),
            "case_snippets_json": json.dumps(res["case_snippets"], ensure_ascii=False),
        }
        records.append(rec)

df_scan = pd.DataFrame.from_records(records)
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_scan.to_csv(OUT_FILE, index=False, encoding="utf-8")

print(f"✅ Scanned {scanned:,} HTML/TXT files from {len(SOURCES)} sources")
print(f"📝 Wrote results to: {OUT_FILE}")

# preview
display(df_scan[[
    "source", "lang_inferred",
    "found_article", "mentions_comp_article", "first_article_mention",
    "found_keywords", "first_keyword_mention"
]].head(20))



C:\Users\eduar\AppData\Local\Temp\ipykernel_26744\2493555965.py:25: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(text, "lxml") if "lxml" in sys.modules else BeautifulSoup(text, "html.parser")


✅ Scanned 33,650 HTML/TXT files from 3 sources
📝 Wrote results to: C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\scan_mentions.csv


,source,lang_inferred,found_article,mentions_comp_article,first_article_mention,found_keywords,first_keyword_mention
0,c1_juris,None,True,True,ARTICLE 60,False,None
1,c1_juris,None,True,True,ARTICLE 60,False,None
2,c1_juris,None,True,False,ARTICLE 2,False,None
3,c1_juris,None,True,False,ARTICLE 15 OF THE TREATY,False,None
4,c1_juris,None,True,True,ARTICLE 15,False,None
5,c1_juris,None,True,True,ARTICLE 35,False,None
6,c1_juris,None,True,False,ARTICLE 35,False,None
7,c1_juris,None,True,False,ARTICLE 34,False,None
8,c1_juris,None,True,False,ARTICLE 34,False,None
9,c1_juris,None,True,False,ARTICLE 42,False,None


In [71]:
# --- Cell 5: aggregates ---

def _explode_json_series(s: pd.Series):
    return (
        s.dropna()
         .apply(lambda x: json.loads(x))
         .explode()
         .dropna()
         .astype(str)
         .str.strip()
    )

print("Top Article strings:")
try:
    print(_explode_json_series(df_scan["article_matches_json"]).value_counts().head(20))
except Exception as e:
    print("No article matches or column missing:", e)

print("\nTop keyword tokens (cartel/kartell/antitrust):")
try:
    print(_explode_json_series(df_scan["keyword_matches_json"]).str.lower().value_counts().head(20))
except Exception as e:
    print("No keyword matches or column missing:", e)

print("\nTop case numbers (IV/AT):")
try:
    print(
        _explode_json_series(df_scan["case_matches_json"])
        .str.upper()
        .value_counts()
        .head(20)
    )
except Exception as e:
    print("No case matches or column missing:", e)


Top Article strings:
article_matches_json
Article 267 TFEU    9124
Article 1           8650
Article 2           7855
Article 3           6651
Article 4           5789
Article 5           4873
Article 6           4450
Article 7           4173
Article 3(1)        3612
Article 2(1)        3595
Article 8           3551
Article 1(1)        3547
Article 69(2)       3189
Article 4(1)        3088
Article 56          3066
Article 234 EC      2860
Article 6(1)        2788
Article 9           2750
Article 10          2606
Article 11          2410
Name: count, dtype: int64

Top keyword tokens (cartel/kartell/antitrust):
keyword_matches_json
cartel              503
wettbewerb          156
antitrust            39
wettbewerbsrecht      1
Name: count, dtype: int64

Top case numbers (IV/AT):
case_matches_json
IV 31.149      47
COMP 39092     28
IV 31.553      17
IV 33.126      16
IV 31.865      15
COMP 39258     12
COMP 37.956    12
IV 33.133      12
IV 31.043      12
COMP 39.125    11
IV 31.370      1